# Small Frame Signal Injection

This notebook starts with the classic in-memory `Frame` workflow:
create a modest dynamic spectrum, add synthetic radiometer noise,
estimate the local noise level, inject a narrowband drifting signal,
and inspect the resulting spectrum and time series.

The scientific contract we want to preserve is that the injected
signal is represented consistently by the current frame axes,
that SNR helpers use explicit noise statistics when supplied, and
that derived products preserve useful provenance.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
frame = stg.Frame(
    fchans=1024,
    tchans=32,
    df=2.7939677238464355 * u.Hz,
    dt=18.253611008 * u.s,
    fch1=6095.214842353016 * u.MHz,
    ascending=False,
    seed=11,
    source_name="Synthetic small frame",
)
frame.add_noise(x_mean=10, noise_type="chi2")

path = stg.constant_path(
    f_start=frame.get_frequency(frame.fchans // 2),
    drift_rate=1.5 * u.Hz / u.s,
)
f_profile = stg.gaussian_f_profile(width=35 * u.Hz)
noise_config = stg.NoiseEstimationConfig(
    method="sigma_clip",
    context_width=256,
    guard_width=24,
)
stats = frame.estimate_noise_stats(
    path=path,
    f_profile=f_profile,
    auto_bounding=True,
    truncate_below=1e-3,
    config=noise_config,
)
level = frame.get_intensity(snr=30, noise_stats=stats)
stats, level

In [ ]:
signal = frame.add_signal(
    path=path,
    t_profile=stg.constant_t_profile(level=level),
    f_profile=f_profile,
    bp_profile=stg.constant_bp_profile(level=1),
    integrate_path=True,
    integrate_t_profile=True,
    integrate_f_profile=True,
    t_subsamples=8,
    f_subsamples=8,
    auto_bounding=True,
    truncate_below=1e-3,
)

print("signal shape:", signal.shape)
print("nonzero signal pixels:", np.count_nonzero(signal))
print("frame noise stats:", frame.get_noise_stats())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
frame.plot(ftype="fmid", ttype="trel", db=True, colorbar=True)
ax.set_title("Small in-memory frame after signal injection")
display(fig)
plt.close(fig)

In [ ]:
spectrum = frame.spectrum(mode="sum", normalize=True)
time_series = frame.timeseries(mode="mean")

fig, axs = plt.subplots(1, 2, figsize=(12, 3))
plt.sca(axs[0])
spectrum.plot(ftype="fmid")
axs[0].set_title("Sigma-normalized spectrum")
plt.sca(axs[1])
time_series.plot(ttype="trel")
axs[1].set_title("Band-averaged time series")
plt.tight_layout()
display(fig)
plt.close(fig)

print("spectrum derived metadata:")
spectrum.metadata["derived"]